# ICC Design Effect — Effective Sample Size (Post-Analysis)

**Method:** Compute ICC from observed cluster-level data via one-way ANOVA decomposition, then derive the effective sample size.

---

$$\text{ICC} = \frac{\text{MSB} - \text{MSW}}{\text{MSB} + (m_0 - 1) \times \text{MSW}}$$

$$\text{DEFF} = 1 + (\bar{m} - 1) \times \text{ICC}$$

$$N_{\text{effective}} = \frac{N_{\text{raw}}}{\text{DEFF}}$$

---

| Symbol | Meaning |
|--------|---------|
| $\text{MSB}$ | Mean Square Between clusters |
| $\text{MSW}$ | Mean Square Within clusters |
| $m_0$ | Adjusted average cluster size (accounts for unequal sizes) |
| $\bar{m}$ | Simple average cluster size |
| $\text{ICC}$ | Intraclass Correlation Coefficient (clipped to ≥ 0) |
| $\text{DEFF}$ | Design Effect |
| $N_{\text{effective}}$ | Effective sample size after adjusting for clustering |

In [4]:
import json

config = json.load(open("../test_config.json"))
d = config["design"]
planned_n = config["computed"]["planned_n"]

# ==========================================
#  OBSERVED DATA — enter cluster-level results
#  (parallel lists: visitors and conversions per cluster)
# ==========================================
cluster_visitors    = [80, 75, 90, 85, 70, 95, 82, 78, 88, 77]
cluster_conversions = [1,  2,  1,  0,  1,  3,  1,  0,  2,  1]
cluster_labels      = None  # auto-generates "1", "2", ... if None
# ==========================================

print(f"Clusters: {len(cluster_visitors)}  |  Total visitors: {sum(cluster_visitors):,}  |  Total conversions: {sum(cluster_conversions)}")

Clusters: 10  |  Total visitors: 820  |  Total conversions: 12


In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm
from scipy.optimize import brentq

def fleiss_mde(n, p0, z_alpha, z_beta):
    """Solve Fleiss one-sample N equation for delta (exact inverse of sample-size formula).
    Solves in both directions and returns the min (matches the design's direction)."""
    def _solve(sign):
        def residual(delta):
            p1 = p0 + sign * delta
            return (z_alpha * np.sqrt(p0 * (1 - p0)) + z_beta * np.sqrt(p1 * (1 - p1))) ** 2 / delta ** 2 - n
        delta_max = (1 - p0 if sign > 0 else p0) - 1e-10
        return brentq(residual, 1e-8, delta_max)
    return min(_solve(+1), _solve(-1))

def analyze_icc_effective_n(cluster_visitors, cluster_conversions, target_rate, planned_n, cluster_labels=None):
    visitors = np.array(cluster_visitors, dtype=float)
    conversions = np.array(cluster_conversions, dtype=float)
    k = len(visitors)
    if cluster_labels is None:
        cluster_labels = [str(i + 1) for i in range(k)]

    N = visitors.sum()
    rates = conversions / visitors
    p_bar = conversions.sum() / N
    m_bar = N / k

    # --- ANOVA-based ICC ---
    # m0: adjusted average cluster size (for unequal sizes)
    m0 = (N - np.sum(visitors**2) / N) / (k - 1)
    SSB = np.sum(visitors * (rates - p_bar)**2)
    SSW = np.sum(visitors * rates * (1 - rates))
    MSB = SSB / (k - 1)
    MSW = SSW / (N - k)
    icc = max(0, (MSB - MSW) / (MSB + (m0 - 1) * MSW))

    deff = 1 + (m_bar - 1) * icc
    effective_n = N / deff

    # MDE comparison (one-sample z-test)
    alpha = 0.05
    z_alpha = norm.ppf(1 - alpha / 2)
    z_beta = norm.ppf(0.80)
    mde_effective = fleiss_mde(effective_n, target_rate, z_alpha, z_beta)
    mde_planned = fleiss_mde(planned_n, target_rate, z_alpha, z_beta)

    sufficient = effective_n >= planned_n

    # --- Output ---
    print(f"--- ICC POST-ANALYSIS (ANOVA) ---")
    print(f"手法 : ANOVA分解によるICC推定 → デザイン効果 → 有効サンプル数")
    print(f"KPI  : コンバージョン率（CVR）")
    print(f"目標 : {target_rate:.1%}")
    print(f"-" * 50)
    print(f"【観測データ】")
    print(f"  クラスター数         : {k}")
    print(f"  総訪問者数 (raw N)   : {int(N):,}")
    print(f"  総CV数               : {int(conversions.sum()):,}")
    print(f"  全体CVR              : {p_bar:.4f} ({p_bar:.2%})")
    print(f"  平均クラスターサイズ  : {m_bar:.1f}")
    print(f"-" * 50)
    print(f"【ICC推定】")
    print(f"  MSB = {MSB:.6f}  |  MSW = {MSW:.6f}")
    print(f"  m₀ (調整平均) = {m0:.2f}")
    print(f"  ICC = max(0, (MSB - MSW) / (MSB + (m₀-1)×MSW))")
    print(f"      = {icc:.6f}")
    print(f"-" * 50)
    print(f"【デザイン効果】")
    print(f"  DEFF = 1 + ({m_bar:.1f} - 1) × {icc:.6f} = {deff:.4f}")
    print(f"  有効サンプル数 = {int(N):,} / {deff:.4f} = {int(np.floor(effective_n)):,}")
    print(f"  計画サンプル数 = {planned_n:,}")
    print(f"-" * 50)
    print(f"【MDE比較】")
    print(f"  有効N基準 MDE : {mde_effective:.2%}")
    print(f"  計画N基準 MDE : {mde_planned:.2%}")
    print(f"-" * 50)
    if sufficient:
        print(f"判定 : SUFFICIENT — 有効サンプル数 ({int(np.floor(effective_n)):,}) ≥ 計画 ({planned_n:,})")
        print(f"解釈 : クラスタリングを考慮しても十分なサンプル数を確保")
    else:
        print(f"判定 : INSUFFICIENT — 有効サンプル数 ({int(np.floor(effective_n)):,}) < 計画 ({planned_n:,})")
        print(f"解釈 : クラスタリングにより検出力が低下。MDE が {mde_effective:.2%} に拡大（計画: {mde_planned:.2%}）")
    print(f"-" * 50)

    # --- Chart 1: Per-cluster CVR dot plot ---
    fig1 = go.Figure()
    show_labels = k <= 30
    fig1.add_trace(go.Scatter(
        x=cluster_labels if show_labels else list(range(k)),
        y=rates, mode='markers',
        name='Cluster CVR',
        marker=dict(color='#1f77b4', size=10)))
    fig1.add_hline(y=p_bar, line_dash="dash", line_color="#d62728",
        annotation_text=f"Overall CVR = {p_bar:.2%}", annotation_position="top left")
    fig1.add_hline(y=target_rate, line_dash="dot", line_color="gray",
        annotation_text=f"Target = {target_rate:.1%}", annotation_position="bottom left")

    fig1.update_layout(
        title=dict(text=f"<b>Per-Cluster CVR</b><br>{k} clusters | ICC = {icc:.4f}", font=dict(size=20)),
        xaxis_title="Cluster" if show_labels else "Cluster Index",
        yaxis_title="CVR",
        yaxis=dict(tickformat=".1%", showgrid=True, gridcolor='#eee'),
        xaxis=dict(showgrid=True, gridcolor='#eee'),
        template="plotly_white", plot_bgcolor='rgba(0,0,0,0)')
    if not show_labels:
        fig1.update_xaxes(tickmode='auto', nticks=10)
    fig1.show()

    # --- Chart 2: Raw N vs Effective N vs Planned N ---
    labels = ["Raw N", "Effective N", "Planned N"]
    values = [int(N), int(np.floor(effective_n)), planned_n]
    colors = ["#1f77b4", "#d62728", "#ff7f0e"]

    fig2 = go.Figure()
    fig2.add_trace(go.Bar(x=labels, y=values,
        marker_color=colors, text=[f"{v:,}" for v in values], textposition='outside'))
    fig2.update_layout(
        title=dict(text=f"<b>Sample Size Comparison</b><br>DEFF = {deff:.4f} | ICC = {icc:.6f}",
            font=dict(size=20)),
        yaxis_title="Sample Size",
        yaxis=dict(showgrid=True, gridcolor='#eee'),
        xaxis=dict(showgrid=True, gridcolor='#eee'),
        template="plotly_white", plot_bgcolor='rgba(0,0,0,0)')
    fig2.show()

In [6]:
analyze_icc_effective_n(
    cluster_visitors=cluster_visitors,
    cluster_conversions=cluster_conversions,
    target_rate=d["target_rate"],
    planned_n=planned_n,
    cluster_labels=cluster_labels)

--- ICC POST-ANALYSIS (ANOVA) ---
手法 : ANOVA分解によるICC推定 → デザイン効果 → 有効サンプル数
KPI  : コンバージョン率（CVR）
目標 : 2.0%
--------------------------------------------------
【観測データ】
  クラスター数         : 10
  総訪問者数 (raw N)   : 820
  総CV数               : 12
  全体CVR              : 0.0146 (1.46%)
  平均クラスターサイズ  : 82.0
--------------------------------------------------
【ICC推定】
  MSB = 0.008999  |  MSW = 0.014498
  m₀ (調整平均) = 81.93
  ICC = max(0, (MSB - MSW) / (MSB + (m₀-1)×MSW))
      = 0.000000
--------------------------------------------------
【デザイン効果】
  DEFF = 1 + (82.0 - 1) × 0.000000 = 1.0000
  有効サンプル数 = 820 / 1.0000 = 820
  計画サンプル数 = 1,283
--------------------------------------------------
【MDE比較】
  有効N基準 MDE : 1.37%
  計画N基準 MDE : 1.10%
--------------------------------------------------
判定 : INSUFFICIENT — 有効サンプル数 (820) < 計画 (1,283)
解釈 : クラスタリングにより検出力が低下。MDE が 1.37% に拡大（計画: 1.10%）
--------------------------------------------------
